# 📚 Kho sách Chùa An Lạc & Đền Thanh Vân
Bấm ▶ ở **từng ô theo thứ tự**. Làm được hoàn toàn trên điện thoại.

| Ô | Việc | Cần khoá R2? |
|---|---|---|
| 1 | Đăng nhập Google, chuẩn bị | không |
| 2 | Sao lưu **toàn bộ** thư viện vào kho R2, vùng riêng `luu-tru/` (game không bao giờ đọc vùng này) | có |
| 2b | *(tuỳ chọn)* Sao lưu vào Google Drive của bạn thay vì R2 | không |
| 3 | Tạo bảng **"Danh mục sách – đánh dấu quyền"** trong Google Drive của bạn | không |
| 4 | Đưa những cuốn đã đánh dấu `x` lên vùng `cong-khai/`, game sẽ mở bản này | có |

🔒 Khoá R2 chỉ nhập vào ô ẩn khi được hỏi. **Không gửi khoá qua chat**, không lưu vào tệp.
Ô nào bị ngắt giữa chừng thì bấm ▶ lại: tệp đã có sẽ được bỏ qua, chạy tiếp chỗ còn thiếu.

In [ ]:
#@title 1 · Đăng nhập Google & chuẩn bị
ACCOUNT_ID = '7391866ef46f5ed5d2814c310a8a53f1'   # Cloudflare Account ID (không phải bí mật)
BUCKET = 'thu-vien-sach'
SRC_ID = '1VbpbVymXXPDv0A5239L9hEnsHXiDrZG2'       # thư mục thư viện trên Google Drive
CSV_URL = 'https://raw.githubusercontent.com/doviettu/like/ccr-c1e2e7ed-k0ue6s/docs/danh-muc-sach.csv'
SHEET_NAME = 'Danh mục sách – đánh dấu quyền'

import io, csv, getpass, subprocess, sys, time, urllib.request
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'boto3'], check=True)
import boto3
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
drv = build('drive', 'v3', cache_discovery=False)
FOLDER = 'application/vnd.google-apps.folder'
TYPES = {'.pdf': 'application/pdf', '.epub': 'application/epub+zip', '.webp': 'image/webp', '.json': 'application/json', '.txt': 'text/plain; charset=utf-8'}
_s3 = None

def s3():
    global _s3
    if _s3 is None:
        _s3 = boto3.client('s3', endpoint_url=f'https://{ACCOUNT_ID}.r2.cloudflarestorage.com', region_name='auto',
                           aws_access_key_id=getpass.getpass('R2 Access Key ID: ').strip(),
                           aws_secret_access_key=getpass.getpass('R2 Secret Access Key: ').strip())
        _s3.head_bucket(Bucket=BUCKET)  # báo lỗi ngay nếu khoá sai hoặc sai tên bucket
        print('✓ Đã nối kho R2', BUCKET)
    return _s3

def call(req, tries=6):
    for k in range(tries):
        try: return req.execute()
        except HttpError as e:
            if int(e.resp.status) in (403, 429, 500, 502, 503) and k < tries - 1: time.sleep(2 ** k); continue
            raise

def ls(pid):
    tok = None
    while True:
        r = call(drv.files().list(q=f"'{pid}' in parents and trashed=false", pageSize=1000, pageToken=tok,
                                  fields='nextPageToken, files(id,name,mimeType,size)', supportsAllDrives=True, includeItemsFromAllDrives=True))
        yield from r.get('files', [])
        tok = r.get('nextPageToken')
        if not tok: return

def r2_keys(prefix):
    keys, kw = set(), {'Bucket': BUCKET, 'Prefix': prefix}
    while True:
        r = s3().list_objects_v2(**kw); keys |= {o['Key'] for o in r.get('Contents', [])}
        if not r.get('IsTruncated'): return keys
        kw['ContinuationToken'] = r['NextContinuationToken']

def r2_put(fid, key, name, stat):
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, drv.files().get_media(fileId=fid, supportsAllDrives=True), chunksize=8 * 1024 * 1024)
    done = False
    while not done: _, done = dl.next_chunk()
    ext = '.' + name.rsplit('.', 1)[-1].lower() if '.' in name else ''
    s3().put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue(), ContentType=TYPES.get(ext, 'application/octet-stream'))
    stat['up'] += 1; stat['bytes'] += buf.tell()
print('✓ Sẵn sàng. Bấm ▶ ở ô tiếp theo.')

In [ ]:
#@title 2 · Sao lưu TOÀN BỘ thư viện vào R2 (vùng riêng luu-tru/)
stat = {'up': 0, 'skip': 0, 'err': 0, 'bytes': 0}
have = r2_keys('luu-tru/')
def walk(src, path):
    for f in ls(src):
        if f['mimeType'] == FOLDER: walk(f['id'], path + f['name'] + '/'); continue
        if f['mimeType'].startswith('application/vnd.google-apps'): continue
        key = 'luu-tru/' + path + f['name']
        if key in have: stat['skip'] += 1; continue
        try: r2_put(f['id'], key, f['name'], stat)
        except Exception as e: stat['err'] += 1; print('Lỗi:', key, '→', e)
        if stat['up'] and stat['up'] % 50 == 0: print(f"… {stat['up']} tệp · {stat['bytes'] / 1e9:.2f} GB")
walk(SRC_ID, '')
print(f"XONG: {stat['up']} tệp mới ({stat['bytes'] / 1e9:.2f} GB), {stat['skip']} đã có, {stat['err']} lỗi.")

In [ ]:
#@title 2b · (Tuỳ chọn) Sao lưu vào Google Drive của bạn — Google tự sao chép, không tốn mạng
DEST_NAME = 'Thu vien kinh sach - sao luu'
st = {'copied': 0, 'skipped': 0, 'errors': 0}
def mkdir(name, parent):
    body = {'name': name, 'mimeType': FOLDER}
    if parent: body['parents'] = [parent]
    return call(drv.files().create(body=body, fields='id'))['id']
r = call(drv.files().list(q=f"name='{DEST_NAME}' and mimeType='{FOLDER}' and 'root' in parents and trashed=false", fields='files(id)'))
root = r['files'][0]['id'] if r.get('files') else mkdir(DEST_NAME, None)
def mirror(src, dst):
    have = {(f['name'], f['mimeType'] == FOLDER): f['id'] for f in ls(dst)}
    for f in ls(src):
        isdir = f['mimeType'] == FOLDER
        if isdir: mirror(f['id'], have.get((f['name'], True)) or mkdir(f['name'], dst)); continue
        if (f['name'], False) in have: st['skipped'] += 1; continue
        try: call(drv.files().copy(fileId=f['id'], body={'name': f['name'], 'parents': [dst]}, fields='id', supportsAllDrives=True)); st['copied'] += 1
        except HttpError as e: st['errors'] += 1; print('Lỗi:', f['name'], e.resp.status)
mirror(SRC_ID, root)
print(f"XONG: {st['copied']} tệp mới, {st['skipped']} đã có, {st['errors']} lỗi → Drive của tôi/{DEST_NAME}")

In [ ]:
#@title 3 · Tạo bảng "Danh mục sách – đánh dấu quyền" trong Google Drive của bạn
r = call(drv.files().list(q=f"name='{SHEET_NAME}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false", fields='files(id)'))
if r.get('files'):
    sid = r['files'][0]['id']; print('Bảng đã có sẵn (không ghi đè chỗ bạn đã đánh dấu).')
else:
    data = urllib.request.urlopen(CSV_URL).read()
    media = MediaIoBaseUpload(io.BytesIO(data), mimetype='text/csv', resumable=False)
    sid = call(drv.files().create(body={'name': SHEET_NAME, 'mimeType': 'application/vnd.google-apps.spreadsheet'}, media_body=media, fields='id'))['id']
    print('✓ Đã tạo bảng.')
print('Mở bảng:', f'https://docs.google.com/spreadsheets/d/{sid}/edit')
print('Ghi x vào cột "duoc_phep" cho cuốn bạn CHẮC CHẮN được phép phát hành lại, ghi căn cứ ở cột "can_cu_quyen". Xong thì chạy ô 4.')

In [ ]:
#@title 4 · Đưa những cuốn đã đánh dấu x lên kho (vùng cong-khai/) — game sẽ mở bản này
r = call(drv.files().list(q=f"name='{SHEET_NAME}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false", fields='files(id)'))
if not r.get('files'): raise SystemExit('Chưa có bảng — hãy chạy ô 3 trước.')
text = drv.files().export(fileId=r['files'][0]['id'], mimeType='text/csv').execute().decode('utf-8-sig')
rows = [x for x in csv.DictReader(io.StringIO(text)) if (x.get('duoc_phep') or '').strip()]
print(f'{len(rows)} cuốn được đánh dấu được phép.')
stat = {'up': 0, 'skip': 0, 'err': 0, 'bytes': 0}
have = r2_keys('cong-khai/')
for x in rows:
    fid = x['id'].strip()
    for f in ls(fid):
        ext = '.' + f['name'].rsplit('.', 1)[-1].lower() if '.' in f['name'] else ''
        if ext not in ('.pdf', '.epub'): continue
        key = f'cong-khai/{fid}{ext}'
        if key in have: stat['skip'] += 1; continue
        try: r2_put(f['id'], key, f['name'], stat); print('✓', x['ten'], ext)
        except Exception as e: stat['err'] += 1; print('Lỗi:', x['ten'], '→', e)
print(f"XONG: {stat['up']} tệp mới, {stat['skip']} đã có, {stat['err']} lỗi. Khoảng 5 phút sau game sẽ mở các cuốn này từ kho riêng.")
print('Kiểm tra: https://bodetam.doviettudn.workers.dev/sach/danh-sach')